# Pedestrain Detection By CityPerson Dataset

Using pretrained model Yolo-v5, Yolo-v8 and Yolo-v11 for pedestrain detection (Object detection)

In [ ]:
# Import required libraries

import os
import numpy as np
import cv2
import json
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from PIL import Image
from collections import Counter
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.utils import Sequence
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

## Convert And Copy Images
This function converts .tif images in a source directory to .jpg format and saves them in a specified destination directory while preserving folder structure.



In [ ]:
def convert_and_copy_images(src_dir, dest_dir):
    # Create the destination directory if it doesn't exist
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)

    # Walk through all folders and subfolders in the source directory
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            # Process only .tif files
            if file.lower().endswith('.tif'):
                # Full path to the .tif image
                file_path = os.path.join(root, file)

                # Open the .tif image and convert it to .jpg
                with Image.open(file_path) as img:
                    # Change the file extension to .jpg
                    new_file_name = os.path.splitext(file)[0] + '.jpg'
                    new_file_path = os.path.join(dest_dir, new_file_name)

                    # Convert and save as .jpg in the destination directory
                    img.convert("RGB").save(new_file_path, "JPEG")
# Example usage
source_folder = '/kaggle/input/city-persone/gtFinePanopticParts_trainval/gtFinePanopticParts/train'
destination_folder = '/kaggle/working/gtFinePanopticParts/train/images'
convert_and_copy_images(source_folder, destination_folder)

In [ ]:
def convert_and_copy_images(src_dir, dest_dir):
    # Create the destination directory if it doesn't exist
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)

    # Walk through all folders and subfolders in the source directory
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            # Process only .tif files
            if file.lower().endswith('.tif'):
                # Full path to the .tif image
                file_path = os.path.join(root, file)

                # Open the .tif image and convert it to .jpg
                with Image.open(file_path) as img:
                    # Change the file extension to .jpg
                    new_file_name = os.path.splitext(file)[0] + '.jpg'
                    new_file_path = os.path.join(dest_dir, new_file_name)

                    # Convert and save as .jpg in the destination directory
                    img.convert("RGB").save(new_file_path, "JPEG")
# Example usage
source_folder = '/kaggle/input/city-persone/gtFinePanopticParts_trainval/gtFinePanopticParts/val'
destination_folder = '/kaggle/working/gtFinePanopticParts/val/images'
convert_and_copy_images(source_folder, destination_folder)

## Convert json files To Yolo Model

This function converts JSON annotations containing bounding box coordinates into YOLO format text files. It normalizes the coordinates and assigns the appropriate class ID based on a predefined class mapping.

In [ ]:
# Mapping of class labels to YOLOv5 class IDs
class_mapping = {
    "pedestrian": 0,
    "rider": 1,
    "sitting person": 2,
    "person group": 3,
    "person (other)": 4
}

def convert_bbox_to_yolo(bbox, img_width, img_height):
    """
    Convert bounding box coordinates from [x_min, y_min, width, height]
    to YOLO format (x_center, y_center, width, height) normalized to image size.
    """
    x_min, y_min, width, height = bbox
    x_center = (x_min + width / 2) / img_width
    y_center = (y_min + height / 2) / img_height
    w = width / img_width
    h = height / img_height
    return x_center, y_center, w, h

def convert_json_to_yolo_txt(json_path, output_dir):
    """
    Convert the JSON annotation to YOLO text file format.
    """
    with open(json_path, 'r') as file:
        data = json.load(file)

    img_width = data["imgWidth"]
    img_height = data["imgHeight"]

    # Prepare the output filename based on the input JSON file
    txt_filename = os.path.splitext(os.path.basename(json_path))[0] + ".txt"
    txt_path = os.path.join(output_dir, txt_filename)

    with open(txt_path, 'w') as txt_file:
        for obj in data["objects"]:
            label = obj["label"]
            if label not in class_mapping:
                continue  # Skip labels that aren't in the class_mapping

            class_id = class_mapping[label]
            bbox = obj["bbox"]

            # Convert bounding box to YOLO format
            x_center, y_center, w, h = convert_bbox_to_yolo(bbox, img_width, img_height)

            # Write to the YOLO formatted .txt file
            txt_file.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

def process_json_files_in_dir(src_dir, output_dir):
    """
    Walk through each folder and subfolder in src_dir, and process all JSON files.
    Convert them to YOLO text format.
    """
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            if file.endswith('.json'):
                json_path = os.path.join(root, file)
                convert_json_to_yolo_txt(json_path, output_dir)

# Define input and output directories
src_dir = '/kaggle/input/city-persone/gtBbox_cityPersons_trainval/gtBboxCityPersons/val'
output_dir = '/kaggle/working/gtFinePanopticParts/val/labels'

# Make sure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Process all JSON files in the source directory and its subdirectories
process_json_files_in_dir(src_dir, output_dir)

## Process json Files To Yolo Model

This script processes JSON annotation files containing bounding box data, converts them to the YOLO format, and saves them as .txt files. It ensures correct normalization and class mapping for training object detection models.

In [ ]:
# Mapping of class labels to YOLOv5 class IDs
class_mapping = {
    "pedestrian": 0,
    "rider": 1,
    "sitting person": 2,
    "person group": 3,
    "person (other)": 4
}

def convert_bbox_to_yolo(bbox, img_width, img_height):
    """
    Convert bounding box coordinates from [x_min, y_min, width, height]
    to YOLO format (x_center, y_center, width, height) normalized to image size.
    """
    x_min, y_min, width, height = bbox
    x_center = (x_min + width / 2) / img_width
    y_center = (y_min + height / 2) / img_height
    w = width / img_width
    h = height / img_height
    return x_center, y_center, w, h

def convert_json_to_yolo_txt(json_path, output_dir):
    """
    Convert the JSON annotation to YOLO text file format.
    """
    with open(json_path, 'r') as file:
        data = json.load(file)

    img_width = data["imgWidth"]
    img_height = data["imgHeight"]

    # Prepare the output filename based on the input JSON file
    txt_filename = os.path.splitext(os.path.basename(json_path))[0] + ".txt"
    txt_path = os.path.join(output_dir, txt_filename)

    with open(txt_path, 'w') as txt_file:
        for obj in data["objects"]:
            label = obj["label"]
            if label not in class_mapping:
                continue  # Skip labels that aren't in the class_mapping

            class_id = class_mapping[label]
            bbox = obj["bbox"]

            # Convert bounding box to YOLO format
            x_center, y_center, w, h = convert_bbox_to_yolo(bbox, img_width, img_height)

            # Write to the YOLO formatted .txt file
            txt_file.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

def process_json_files_in_dir(src_dir, output_dir):
    """
    Walk through each folder and subfolder in src_dir, and process all JSON files.
    Convert them to YOLO text format.
    """
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            if file.endswith('.json'):
                json_path = os.path.join(root, file)
                convert_json_to_yolo_txt(json_path, output_dir)

# Define input and output directories
src_dir = '/kaggle/input/city-persone/gtBbox_cityPersons_trainval/gtBboxCityPersons/train'
output_dir = '/kaggle/working/gtFinePanopticParts/train/labels'

# Make sure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Process all JSON files in the source directory and its subdirectories
process_json_files_in_dir(src_dir, output_dir)

## Rename Annotation Files

This function renames all .txt annotation files in a given directory, replacing occurrences of "BboxCityPersons" with "FinePanopticParts" to ensure consistency in naming conventions.

In [ ]:
def rename_files_in_dir(output_dir):
    """
    Rename all .txt files in the output_dir by replacing 'BboxCityPersons' with 'FinePanopticParts'.
    """
    # Loop through all files in the directory
    for file_name in os.listdir(output_dir):
        # Check if the file is a .txt file
        if file_name.endswith('.txt'):
            # Replace 'BboxCityPersons' with 'FinePanopticParts' in the file name
            new_name = file_name.replace('BboxCityPersons', 'FinePanopticParts')

            # Get the full path of the old and new file names
            old_file_path = os.path.join(output_dir, file_name)
            new_file_path = os.path.join(output_dir, new_name)

            # Rename the file
            os.rename(old_file_path, new_file_path)

# Define the output directory where the text files are located
output_dir = '/kaggle/working/gtFinePanopticParts/val/labels'  # Replace with the actual path to your folder
output_dir1 = '/kaggle/working/gtFinePanopticParts/train/labels'  # Replace with the actual path to your folder

# Rename all .txt files in the directory
rename_files_in_dir(output_dir)
rename_files_in_dir(output_dir1)

## Load And Split Dataset

This function loads images and their corresponding YOLO-format bounding box annotations, normalizes the coordinates, and splits the validation dataset into 90% validation and 10% test sets.

In [ ]:
from sklearn.model_selection import train_test_split


def load_images_and_annotations(image_folder, annotation_folder, file_extension='.jpg'):
    images = {}
    annotations = {}
    
    for filename in os.listdir(image_folder):
        if filename.endswith(file_extension):
            img_path = os.path.join(image_folder, filename)
            annotation_path = os.path.join(annotation_folder, filename.replace(file_extension, '.txt'))
            
            # Read image
            img = cv2.imread(img_path)
            if img is not None:
                images[filename] = img
                
                # Read bounding box annotations
                with open(annotation_path, 'r') as file:
                    bboxes = []
                    for line in file:
                        class_label, x_center, y_center, width, height = map(float, line.strip().split())

                        # Denormalize coordinates
                        x_center = int(x_center * img.shape[1])
                        y_center = int(y_center * img.shape[0])
                        width = int(width * img.shape[1])
                        height = int(height * img.shape[0])

                        # Convert to (x, y, width, height)
                        x = max(0, x_center - width // 2)
                        y = max(0, y_center - height // 2)
                        width = min(width, img.shape[1] - x)
                        height = min(height, img.shape[0] - y)

                        if width > 0 and height > 0:
                            bboxes.append((x, y, width, height, int(class_label)))

                    annotations[filename] = bboxes

    return images, annotations

# Define dataset paths
train_images, train_annotations = load_images_and_annotations('/kaggle/working/gtFinePanopticParts/train/images', '/kaggle/working/gtFinePanopticParts/train/labels')
val_images, val_annotations = load_images_and_annotations('/kaggle/working/gtFinePanopticParts/val/images', '/kaggle/working/gtFinePanopticParts/val/labels')

print(f"Loaded {len(train_images)} training images and {len(val_images)} validation images.")

# Splitting validation set into 90% validation and 10% test
val_images, test_images, val_annotations, test_annotations = train_test_split(
    list(val_images.items()), list(val_annotations.items()), test_size=0.1, random_state=42
)

# Convert back to dictionaries
val_images = dict(val_images)
test_images = dict(test_images)
val_annotations = dict(val_annotations)
test_annotations = dict(test_annotations)

print(f"Updated validation set: {len(val_images)} images")
print(f"Test set: {len(test_images)} images")